## Transform Drivers Data
- Read bronze driver table
- Keep only the columns required for analytics(Drop url column)
- Standardise column name using snakecase(driverId->driver_id)
- Rename columns to make them more meaningful(name->constructor_name)
- Remove duplicate records.
- Transform values of column nationality to Title Case
- Write the transformed data to silver driver table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/03.silver-help

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.drivers"
silver_table=f"{catalog_name}.{silver_schema}.drivers"

In [0]:
import pyspark.sql.functions as F

### Step 1 Read bronze Drivers table

In [0]:
drivers_df=(
    spark.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
)


### Srep 2 -Keep only the columns required for analytics(Drop url column)

In [0]:
#flexible   `

drivers_selected_df=(
    drivers_df.drop("url")
)

### Step 3 - Stamdardise column name
- Standardise column name using snakecase(driverId->driver_id)


In [0]:
drivers_renamed_df=(
    drivers_selected_df.withColumnRenamed("driverId","driver_id")
                       .withColumnRenamed("dateOfBirth","date_of_birth")
                )
#display(drivers_renamed_df)

### Step 4 Concatenate name.givenName and name.familyName to Create a new column called as driver_name

In [0]:
drivers_concatenated_df=(
    drivers_renamed_df
    .withColumn("driver_name",
                F.initcap(F.concat_ws(" ",F.col("name.givenName"),F.col("name.familyName"))))
    .drop("name")
    
)

#dropduplicates 

In [0]:
display(drivers_concatenated_df)

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,batch_id,driver_name
abate,1932-07-10,italian,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Carlo Abate
abecassis,1913-03-21,british,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,George Abecassis
acheson,1957-11-27,british,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kenny Acheson
adams,1969-11-19,belgian,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Philippe Adams
ader,1913-12-15,american,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Walt Ader
adolff,1921-11-05,german,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kurt Adolff
agabashian,1913-08-21,american,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Fred Agabashian
ahrens,1940-04-19,german,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kurt Ahrens
aitken,1995-09-23,british,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Jack Aitken
albers,1979-04-16,dutch,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Christijan Albers


### Step -5 Remove duplicates

In [0]:
#dropduplicates - you can give column name and distinct will be applied on that particular column only
drivers_distinct_df=drivers_concatenated_df.dropDuplicates(["driver_id"])
#display(drivers_distinct_df)


### Step -6 Transform values of column nationality to Title Case

In [0]:
drivers_final_df=(
    drivers_distinct_df
    .withColumn("nationality",F.initcap(F.col("nationality")))
)
#display(drivers_final_df)

### Step-7 Write the transformed data to silver drivers table

In [0]:
write_to_silver(
    drivers_final_df,
    silver_table,
    merge_condition="s.driver_id=t.driver_id",
    columns_to_update=[
        "driver_id",
        "date_of_birth",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id",
        "driver_name"
    ]
)

In [0]:
display(spark.read.table(silver_table))

driver_id,date_of_birth,nationality,ingestion_timestamp,source_file,batch_id,driver_name,created_timestamp,updated_timestamp
ahrens,1940-04-19,German,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Kurt Ahrens,2026-10-20T11:32:28.179Z,2026-10-20T11:32:47.521Z
barilla,1961-04-20,Italian,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Paolo Barilla,2026-10-20T11:32:28.179Z,2026-10-20T11:32:47.521Z
bayol,1914-02-28,French,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Élie Bayol,2026-10-20T11:32:28.179Z,2026-10-20T11:32:47.521Z
birger,1924-01-07,Argentine,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Pablo Birger,2026-10-20T11:32:28.179Z,2026-10-20T11:32:47.521Z
borgudd,1946-11-25,Swedish,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Slim Borgudd,2026-10-20T11:32:28.179Z,2026-10-20T11:32:47.521Z
branca,1916-09-15,Swiss,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Toni Branca,2026-10-20T11:32:28.179Z,2026-10-20T11:32:47.521Z
brown,1949-12-24,Australian,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Warwick Brown,2026-10-20T11:32:28.179Z,2026-10-20T11:32:47.521Z
christie,1924-04-04,American,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Bob Christie,2026-10-20T11:32:28.179Z,2026-10-20T11:32:47.521Z
clark,1936-03-04,British,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,Jim Clark,2026-10-20T11:32:28.179Z,2026-10-20T11:32:47.521Z
george_connor,1906-08-16,American,2026-10-20T07:33:24.721Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01,George Connor,2026-10-20T11:32:28.179Z,2026-10-20T11:32:47.521Z
